### Setting up Environment and testing LLM response

In [3]:
import openai
import minsearch
import requests

In [4]:
from dotenv import load_dotenv
load_dotenv(override=True) # If True .env is loaded

True

##### Load the keys to openAI 

In [5]:
from openai import OpenAI
openai_client = OpenAI()

In [51]:
# Create a simple prompt
prompt = [{'role': 'user',
          'content': "What is the capital of\
              Yugoslavia?"}]


# Send a LLM request
response = openai_client.chat.completions.create(
                messages=prompt,
                model='gpt-5.4-mini'
)



In [52]:
# Print the response
print(response.choices[0].message.content)

The capital of Yugoslavia was **Belgrade**.


In [8]:
!openai --version  # Checking openai version

openai 2.3.0


In [9]:
# Convert the llm sending script to a function
def llm_response(prompt):
    llm_response = openai.chat.completions.create(
        model='gpt-5.4-mini',
        messages=[{'role': 'user',
          'content': prompt}]
    )
    return llm_response.choices[0].message.content

In [10]:
# Sending another llm request
print(llm_response("hey whats up, bitch?!"))

Hey — I’m here to help, but let’s keep it respectful. What can I do for you?


In [11]:
# Sending another llm request
question = "Can I join the course"
answer = llm_response(question)
print(answer)

Yes — you can join the course if you meet the enrollment requirements.

If you want, I can help you figure out:
- whether you’re eligible,
- how to apply or register,
- what documents you may need,
- or whether there are any deadlines.

If you meant a specific course, send me its name and I’ll check with you.


Right now the LLM is passing a very generic response and not relevant to the course that I am looking for

### Building a simple RAG system

We create a variable called `context` where we copy and paste the FAQs of the datatalks website and pass that to the LLM as context for more informed answer

In [12]:
# Defining the context variable

context = '''I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.
edit on GitHub
#
How should I start the course and follow the weekly workflow?

Start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the course management platform.

A typical workflow is:

    Watch the lesson videos.
    Work through the lesson notebooks/code.
    Read the homework instructions on GitHub.
    Submit answers through the course platform before the deadline.

Homework is similar to the lesson flow, but uses a different dataset or slightly different task.
edit on GitHub
#
Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?

When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

    First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
    Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!

edit on GitHub
#
Certificate: Can I follow the course in a self-paced mode and get a certificate?

No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.
edit on GitHub
#
I missed the first homework - can I still get a certificate?

Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.
edit on GitHub
#
Homework: Why does the content keep changing?

If the homework title contains [DRAFT], it means the homework is not ready yet.

The homework is ready only when both are true:

    The homework form is open on the course management platform.
    The homework title does not contain [DRAFT].

Until then, the content can still change. Working on the material or homework in advance is at your own risk, because the final version can be different.
edit on GitHub
#
When will the course be offered next?

Summer 2027.
edit on GitHub
#
Are there any lectures/videos? Where are they?

Use the LLM Zoomcamp GitHub repository as the main entry point.

Open the lesson folders in the repo. Each lesson page has the relevant videos linked at the top.

Lesson pages have the relevant video linked at the top

When in doubt, follow the GitHub repo first, because it is easier to keep updated than the YouTube playlist.
edit on GitHub
#
Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?

Use the LLM Zoomcamp course management platform.

It contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.
edit on GitHub
#
Are there live sessions or office hours for each module?

There are no separate live sessions for every module by default. Module materials are pre-recorded and available in the course repo.

Live sessions are announced separately when they happen. If you are stuck, ask your question in Slack and follow the asking questions guidelines.

Optional extra support is available through AI Shipping Labs, a paid community that includes regular Zoom office hours and additional structure. This is optional; the DataTalks.Club course content remains free.
edit on GitHub
#
Can I use Bluesky for learning in public credits?

Yes. Bluesky posts can be used for learning in public credits.
edit on GitHub
#
Where is the LLM Zoomcamp Telegram channel?

The Telegram channel is https://t.me/llm_zoomcamp.

Use it for announcements. For technical discussion and questions, use the course Slack channel.
'''

In [13]:
question = "I have note submited the homeworks, can I still get the certificate"

In [14]:
# adding question and context to the prompt

prompt = f'''
Your task is to answer questions from the course \
participants based on the provided context.

Use the context to find relevant information and \
provide accurate answers. If the answer is not found in the context, respond with I don't know

Question:
{question}

Context:
{context}

'''


In [15]:
# Send the message to llm again

answer = llm_response(prompt)

In [16]:
print(answer)

Yes — if you’ve missed the homework, that’s okay. You can still get the certificate as long as you pass the Capstone project. Homework is not mandatory, though it is recommended.


As we can see that the LLM was able to identify the context accurately and give the right answer as per the expected domain.

The second answer is lot better than the generic answer that the 1st response gave as the LLM had the context before answering the question

---

In RAG, there are necessarily 3 steps:
1. Search the question
2. Build the prompt
3. Send the prompt to llm for final answer

### Building the search function

In [17]:
# function to perform rag

def rag(question):
   # search the question in DB
   # Build the prompt
   # Send the prompt to LLM
   pass

In [18]:
all_courses_url = "https://datatalks.club/faq/json/courses.json"
all_courses_response = requests.get(all_courses_url)
all_courses_response.raise_for_status()
all_courses_data = all_courses_response.json()

print(all_courses_data)


[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 85}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


In [19]:
documents = []
url_path = "https://datatalks.club/faq"
for course in all_courses_data:
    course_url = url_path + course['path']

    # print(course_url)
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [20]:
documents[1100]

{'id': 'ed90e0f589',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Bind for 0.0.0.0:9696 failed: port is already allocated',
 'answer': 'I was getting the following error when I rebuilt the Docker image, although the port was not allocated, and it was working fine.\n\nError message:\n\n```\nError response from daemon: driver failed programming external connectivity on endpoint beautiful_tharp (875be95c7027cebb853a62fc4463d46e23df99e0175be73641269c3d180f7796): Bind for 0.0.0.0:9696 failed: port is already allocated.\n```\n\n\n\nThe issue can be resolved by running the following command:\n\n```bash\ndocker kill $(docker ps -q)\n```\n\nFor more information, refer to the [GitHub issue on Docker for Windows](https://github.com/docker/for-win/issues/2722).'}

In [21]:
from minsearch import Index
index  = Index(
    text_fields = ['question', 'section', 'answer'],
    keyword_fields = ['course']
)

index.fit(documents)

In [22]:
# Let us know search for a question
print(question)
index.search(question)

I have note submited the homeworks, can I still get the certificate


[{'id': '3774a79c13',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Do I need to do the homeworks to get the certificate?',
  'answer': 'No, as long as you complete the peer-reviewed capstone projects on time, you can receive the certificate. You do not need to do the homeworks if you join late, for example.'},
 {'id': '1a4a7bd286',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Are homeworks required to get the certificate?',
  'answer': 'Homeworks are optional for certification, but strongly recommended to check understanding. Certification is based on projects, not homework scores.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not ma

In [23]:
len(index.search(question))

10

By default, the search is returning 10 results

In [24]:
# Let us use a filter to reduce the results to 5
# Use a filter to only return results relevant to llm engineering

search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course':'llm-zoomcamp'},
    num_results=5)

In [25]:
question

'I have note submited the homeworks, can I still get the certificate'

In [26]:
for result in search_results:
    print(result['question'])
    print("---")

I missed the first homework - can I still get a certificate?
---
Certificate: Can I follow the course in a self-paced mode and get a certificate?
---
I just discovered the course. Can I still join?
---
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
---
OpenAI: Do I have to subscribe and pay for Open AI API for this course?
---


In [27]:
# Let us convert the search index into a function

def search(question, course):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course' : course}

    return index.search(question,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

In [28]:
question = "I found about the course just now. Can I still join?"
search_results = search(question, course='llm-zoomcamp')


In [29]:
for result in search_results:
    print(result['question'])

I just discovered the course. Can I still join?
I missed the first homework - can I still get a certificate?
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
Certificate: Can I follow the course in a self-paced mode and get a certificate?
How should I start the course and follow the weekly workflow?


In [30]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

### Building the prompt function

#### INSTRUCTIONS

In [31]:
INSTRUCTIONS = """ 
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

"""

#### USER PROMPT TEMPLATE

In [37]:
USER_PROMPT_TEMPLATE = """

Question:
{question}


Context:
{context}

"""

#### BUILDING THE CONTEXT

In [39]:
def build_context(search_results):
    lines = []
    
    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q: " + doc['question'])
        lines.append("Ans: " + doc['answer'])
        lines.append("")

    return "\n".join(lines).strip()


In [40]:
build_context(search_results=search_results)

'General Course-Related Questions\nQ: I just discovered the course. Can I still join?\nAns: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: I missed the first homework - can I still get a certificate?\nAns: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nAns: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the co

#### BUILDING THE PROMPT

In [45]:
def build_prompt(question, search_results):
    context = build_context(search_results=search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question,
        context = context
    )

    return prompt.strip()


In [56]:
prompt = build_prompt(question, search_results)

### Sending prompt to LLM

In [61]:
response = openai_client.chat.completions.create(
     messages=[{'role': 'user',
                'content': prompt}],

     model='gpt-5.4-mini'

)

In [62]:
response.choices[0].message.content

'Yes — you can still join. You can start the course anytime.\n\nIf you want to receive a certificate, make sure you submit your project while submissions are still open.'

### Anatomy of the response

In [64]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-DuhBuCBomJy8WtW43XoYrgIyMZz2g",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Yes — you can still join. You can start the course anytime.\n\nIf you want to receive a certificate, make sure you submit your project while submissions are still open.",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1782404766,
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 37,
    "prompt_tokens": 522,
    "total_tokens": 559,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_

In [71]:
response.choices[0].message.content

'Yes — you can still join. You can start the course anytime.\n\nIf you want to receive a certificate, make sure you submit your project while submissions are still open.'

The usage counts tell you how many tokens the request consumed

In [73]:
response.usage

CompletionUsage(completion_tokens=37, prompt_tokens=522, total_tokens=559, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

In [80]:
def calculate_prompt_cost(llm_response):
    input_price = 0.75 / 1_000_000
    output_price = 4.50 / 1_000_000

    cost = (
        llm_response.usage.prompt_tokens * input_price +
        llm_response.usage.completion_tokens * output_price
    )

    return cost

### Back to building prompt and sending api call to LLM

In [75]:
message_history = [

    {'role': 'system',
     'content': INSTRUCTIONS},
    {'role': 'user',
     'content': build_prompt(question, search_results)}
]


llm_response = openai_client.chat.completions.create(
    model='gpt-5.4-mini',
    messages= message_history
)

In [78]:
llm_response.choices[0].message.content

'Yes, you can still join. If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [82]:
# Print the cost of the llm api call
print("This API call costed: " + str(calculate_prompt_cost(llm_response)))

This API call costed: 0.0005745


In [85]:
# Analyzing the prompt one more time
message_history

[{'role': 'system',
  'content': ' \nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\n'},
 {'role': 'user',
  'content': 'Question:\nI found about the course just now. Can I still join?\n\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nAns: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: I missed the first homework - can I still get a certificate?\nAns: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.\n\nGeneral Course-Related Questions\nQ: Course: I have registered fo

### Building a function to send LLM response

In [83]:
def llm_response(instructions, user_prompt, model='gpt-5.4-mini'):
    llm_response = openai_client.chat.completions.create(
        messages = [
           { 'role': 'system',
             'content': instructions},
        {'role': 'user',
         'content': user_prompt}
        ],
        model=model
    )

    return llm_response.choices[0].message.content

In [84]:
print(llm_response(instructions=INSTRUCTIONS,
                   user_prompt=prompt))

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.


### BUILDING A FUNCTION FOR RAG PIPELINE

In [88]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query, course='llm-zoomcamp')  # using the search function
    prompt = build_prompt(query, search_results)
    answer = llm_response(INSTRUCTIONS, prompt, model=model)
    return answer

In [89]:
query = "When Can I join the course?"

print(rag(query))

You can join whenever you want. The course materials are available, and you can start learning and submitting homework while the form is open. If you want a certificate, you need to participate with a live cohort and submit your project while submissions are still being accepted.
